# 정보처리기사 SFT 데이터셋 구축 보고서

이 노트북은 정보처리기사 필기/실기 시험 준비 플래너를 위한 SFT 데이터셋 구축 과정을 간단히 기록한다. 목표는 크롤링한 시험 후기에서 준비 기간, 점수, 결과, 약점 과목/범위, 공부 과정을 구조화하고, 모델이 시험 준비 계획을 JSON으로 생성하도록 학습 가능한 `{messages, meta}` 형식의 내부 학습 데이터를 만드는 것이다.

기준일: 2026-06-09  
대상 시험: 정보처리기사 필기/실기


## 1. 산출물 요약

- 공식 시험 정보: `sft_pipeline/data/exam_info/information_processing_engineer.json`
- 필기 1차 구조화: `exam_information_processing_engineer_sft.jsonl`
- 실기 구조화: `exam_information_processing_engineer_practical_sft.jsonl`
- 필기 추가 구조화: `exam_information_processing_engineer_written_expansion_sft.jsonl`
- 불합격/재도전 추가 구조화: `exam_information_processing_engineer_retry_expansion_sft.jsonl`
- 검수용 phases 합본: `exam_information_processing_engineer_all_sft.jsonl`
- 학습 파이프라인용 runtime 변환본: `exam_information_processing_engineer_runtime_sft.jsonl`
- 꼬리질문 synthetic 배치: `exam_information_processing_engineer_followup_sft.jsonl`
- 구조화 출력 보강 배치: `exam_information_processing_engineer_hardening_sft.jsonl`
- 최종 hardened dry-run split: `ipe_hardened_dryrun_train.jsonl`, `ipe_hardened_dryrun_valid.jsonl`
- 정처기 내부 학습 후보 합본: `exam_information_processing_engineer_training_sft.jsonl`

배치별 파일은 검수와 출처 추적을 위해 보존한다. `all_sft`는 원래의 시험 단계(`phases`) 구조를 확인하기 위한 합본이고, 실제 기존 학습 파이프라인에 섞을 정처기 입력 후보는 hardened split을 사용한다.


In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
G = ROOT / "sft_pipeline" / "data" / "generated"

batch_files = [
    G / "exam_information_processing_engineer_sft.jsonl",
    G / "exam_information_processing_engineer_written_expansion_sft.jsonl",
    G / "exam_information_processing_engineer_practical_sft.jsonl",
    G / "exam_information_processing_engineer_retry_expansion_sft.jsonl",
]
combined_file = G / "exam_information_processing_engineer_all_sft.jsonl"
runtime_file = G / "exam_information_processing_engineer_runtime_sft.jsonl"
followup_file = G / "exam_information_processing_engineer_followup_sft.jsonl"
training_file = G / "exam_information_processing_engineer_training_sft.jsonl"


def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


combined = load_jsonl(combined_file)
runtime_samples = load_jsonl(runtime_file)
followup_samples = load_jsonl(followup_file)
training_samples = load_jsonl(training_file)
len(combined), len(runtime_samples), len(followup_samples), len(training_samples)


## 2. 구축 절차

1. Q-Net 공식 정보와 2026 출제기준 PDF 기준으로 시험 정보 JSON을 정리했다.
2. 후기 URL을 배치별로 수집했다.
3. 크롤러가 `robots.txt`를 확인한 뒤 본문을 추출했다.
4. 원문 전체를 학습 데이터에 넣지 않고, 준비 기간/점수/결과/공부 과정/약점만 재서술해 구조화했다.
5. 필기, 실기, 필기 추가, 재도전 배치를 각각 SFT JSONL로 만들었다.
6. 검수용으로 `exam_information_processing_engineer_all_sft.jsonl` 합본을 생성했다.
7. 기존 학습 파이프라인에 연결하기 위해 `phases` 기반 assistant 출력을 `summary_text/todos/calendar_events` 런타임 스키마로 변환했다.
8. 정보가 부족한 시험 요청에서 바로 plan을 만들지 않도록 `follow_up` synthetic 100건을 생성했다.
9. runtime plan 45건과 follow_up 100건을 결합해 정처기 내부 학습 후보 145건을 만들었다.
10. gold distractor 9건과 runtime hardening 80건을 섞어 최종 hardened dry-run split 234건을 구성했다.

크롤 원문은 `data/generated/`에 남기되, 최종 SFT에는 원문 전문을 포함하지 않는다.


In [ ]:
for path in batch_files:
    rows = load_jsonl(path)
    print(f"{path.name}: {len(rows)} rows")

print(f"{combined_file.name}: {len(combined)} rows")

## 3. 데이터 분포

최종 합본은 총 45건이다. 필기 25건, 실기 20건으로 구성되어 있고, 재도전/불합격 맥락이 포함되어 있다.


In [ ]:
def meta_counter(key: str) -> Counter:
    return Counter((row.get("meta") or {}).get(key, "?") for row in combined)


summary = {
    "total": len(combined),
    "exam_part": dict(meta_counter("exam_part")),
    "result": dict(meta_counter("result")),
    "source_batch": dict(meta_counter("source_batch")),
    "provenance": dict(meta_counter("provenance")),
}
summary

In [ ]:
def compact_row(row: dict) -> dict:
    meta = row["meta"]
    return {
        "id": meta["id"],
        "exam_part": meta["exam_part"],
        "result": meta["result"],
        "reported_score": meta.get("reported_score"),
        "time_left_days": meta.get("time_left_days"),
        "daily_hours": meta.get("daily_hours"),
        "source_batch": meta.get("source_batch"),
    }


for row in combined[:5]:
    print(json.dumps(compact_row(row), ensure_ascii=False, indent=2))

In [ ]:
def assistant_kind(row: dict) -> str:
    payload = json.loads(row["messages"][-1]["content"])
    return payload.get("kind", "runtime_plan")


training_distribution = {
    "rows": len(training_samples),
    "provenance": dict(
        Counter(row.get("meta", {}).get("provenance", "?") for row in training_samples)
    ),
    "meta_kind": dict(
        Counter(row.get("meta", {}).get("kind", "plan") for row in training_samples)
    ),
    "assistant_kind": dict(Counter(assistant_kind(row) for row in training_samples)),
}
print(json.dumps(training_distribution, ensure_ascii=False, indent=2))


## 4. 사용한 스크립트

이번 작업에서 작성한 주요 코드는 아래와 같다.

- `sft_pipeline/build/_archive_ipe/structure_ipe_crawl.py`: 필기 1차 크롤 결과 구조화
- `sft_pipeline/build/_archive_ipe/structure_ipe_practical_crawl.py`: 실기 크롤 결과 구조화
- `sft_pipeline/build/_archive_ipe/structure_ipe_written_expansion_crawl.py`: 필기 추가 크롤 결과 구조화
- `sft_pipeline/build/_archive_ipe/structure_ipe_retry_expansion_crawl.py`: 불합격/재도전 중심 필기/실기 혼합 구조화
- `sft_pipeline/build/_archive_ipe/combine_ipe_sft.py`: 배치별 SFT JSONL 합본 생성
- `sft_pipeline/build/_archive_ipe/validate_exam_sft.py`: 정보처리기사 exam SFT 전용 검증
- `sft_pipeline/build/_archive_ipe/convert_exam_phases_to_runtime.py`: `phases` 출력 → 런타임 플랜 스키마 변환
- `sft_pipeline/build/_archive_ipe/build_ipe_followup_sft.py`: 정보 부족 요청용 `follow_up` synthetic 생성
- `sft_pipeline/build/_archive_ipe/combine_ipe_training_sft.py`: runtime plan + follow_up 결합

합본 단계에서는 `meta.id` 기준 중복 제거, `provenance=exam-crawl`, `exam_type`, `source_batch` 메타 보강을 수행한다. 검증 단계에서는 `phases` 기반 시험 플랜 스키마와 공식 과목명·합격 기준 반영 여부를 확인하고, 변환 단계에서는 오늘 할 일만 `todos`, 미래 일정은 `calendar_events`로 분기한다.


In [ ]:
commands = [
    "python3 -m sft_pipeline.build._archive_ipe.structure_ipe_crawl --today 2026-06-09",
    "python3 -m sft_pipeline.build._archive_ipe.structure_ipe_practical_crawl --today 2026-06-09",
    "python3 -m sft_pipeline.build._archive_ipe.structure_ipe_written_expansion_crawl --today 2026-06-09",
    "python3 -m sft_pipeline.build._archive_ipe.structure_ipe_retry_expansion_crawl --today 2026-06-09",
    "python3 -m sft_pipeline.build._archive_ipe.combine_ipe_sft --out sft_pipeline/data/generated/exam_information_processing_engineer_all_sft.jsonl",
    "python3 -m sft_pipeline.build._archive_ipe.validate_exam_sft --in sft_pipeline/data/generated/exam_information_processing_engineer_all_sft.jsonl",
    "python3 -m sft_pipeline.build._archive_ipe.convert_exam_phases_to_runtime --in sft_pipeline/data/generated/exam_information_processing_engineer_all_sft.jsonl --out sft_pipeline/data/generated/exam_information_processing_engineer_runtime_sft.jsonl",
    "python3 -m sft_pipeline.build.lib.validate_dataset --in sft_pipeline/data/generated/exam_information_processing_engineer_runtime_sft.jsonl",
    "python3 -m sft_pipeline.build._archive_ipe.build_ipe_followup_sft --total 100 --today 2026-06-09 --out sft_pipeline/data/generated/exam_information_processing_engineer_followup_sft.jsonl",
    "python3 -m sft_pipeline.build._archive_ipe.combine_ipe_training_sft --out sft_pipeline/data/generated/exam_information_processing_engineer_training_sft.jsonl",
    "python3 -m sft_pipeline.build.lib.validate_dataset --in sft_pipeline/data/generated/exam_information_processing_engineer_training_sft.jsonl",
]

for command in commands:
    print(command)


## 5. 샘플 형식

각 샘플은 `{messages, meta}` 구조다. `messages`는 system/user/assistant 3턴이며, assistant는 시험 준비 플랜 JSON을 출력한다. `meta`에는 출처, 시험 파트, 결과, 점수, 기간, 약점, 공부 과정 요약이 들어간다.


In [ ]:
sample = combined[0]
print("messages roles:", [m["role"] for m in sample["messages"]])
print("meta keys:", sorted(sample["meta"].keys()))
print(
    "assistant plan keys:", sorted(json.loads(sample["messages"][-1]["content"]).keys())
)

In [ ]:
print(json.dumps(sample["meta"], ensure_ascii=False, indent=2)[:2000])

## 6. 품질 점검 코드

아래 셀은 합본 파일의 기본 품질을 점검한다.

- JSONL 파싱 가능 여부
- `meta.id` 중복 여부
- `provenance`, `exam_type`, `source_batch` 존재 여부
- assistant 출력이 JSON plan인지 여부
- 필기/실기 합격 기준 문구 포함 여부


In [ ]:
ids = []

for idx, row in enumerate(combined, start=1):
    meta = row.get("meta") or {}
    ids.append(meta.get("id"))

    assert meta.get("provenance") == "exam-crawl", idx
    assert meta.get("exam_type") in {"정보처리기사 필기", "정보처리기사 실기"}, idx
    assert meta.get("source_batch"), idx
    assert len(row.get("messages", [])) == 3, idx

    plan = json.loads(row["messages"][-1]["content"])
    assert plan.get("kind") == "plan", idx
    assert "summary_text" in plan, idx

    if meta.get("exam_part") == "written":
        assert "과목당 40점 이상" in plan["summary_text"], idx
        assert "평균 60점 이상" in plan["summary_text"], idx
    elif meta.get("exam_part") == "practical":
        assert "정보처리실무" in plan["summary_text"], idx
        assert "60점 이상" in plan["summary_text"], idx
    else:
        raise AssertionError((idx, meta.get("exam_part")))

assert len(ids) == len(set(ids))
print("basic quality checks passed", len(combined))

## 7. 학습 데이터셋 목적

이번 1차 데이터셋의 목적은 정보처리기사 수험자의 실제 준비 맥락을 입력으로 받아, 모델이 시험 파트별 합격 기준과 과목명을 지키면서 실행 가능한 준비 플랜 JSON을 만들도록 학습시키는 것이다. 특히 단순한 합격 후기 요약이 아니라, 준비 기간·점수·합격/불합격 결과·취약 과목·공부 과정을 `meta`로 보존하고 assistant 출력은 구조화된 플랜으로 고정했다.

학습 관점에서 이 데이터는 다음 세 가지 행동을 강화한다.

1. 사용자의 남은 기간과 시험 파트에 따라 개념 정리, 기출, 오답, 총정리 단계를 나누기
2. 필기와 실기의 공식 과목명 및 합격 기준을 계획 설명에 반영하기
3. 불합격/재도전 사례에서는 과락, 실전 연습 부족, 약점 범위 보완을 우선순위에 올리기


## 8. 수집 출처와 근거

데이터는 공식 정보와 후기성 크롤 자료를 분리해 사용했다. 공식 정보는 시험명, 과목명, 합격 기준, 출제기준 기간처럼 바뀌면 안 되는 사실을 고정하는 기준으로 사용했고, 후기성 자료는 준비 기간, 점수, 공부 과정, 실패/재도전 패턴을 추출하는 데 사용했다. 원문 전체는 학습 데이터에 넣지 않고 구조화 요약만 남겼다.


In [ ]:
from urllib.parse import urlparse

source_urls = []
for row in combined:
    meta = row.get("meta", {})
    url = meta.get("source_url") or meta.get("url")
    if url:
        source_urls.append(url)

domains = Counter(urlparse(url).netloc for url in source_urls)
source_summary = {
    "official_info_file": str(info_file),
    "combined_rows": len(combined),
    "rows_with_source_url": len(source_urls),
    "unique_source_domains": len(domains),
    "top_source_domains": domains.most_common(10),
}
print(json.dumps(source_summary, ensure_ascii=False, indent=2))


### 출처 유형 정리

- 공식 기준: Q-Net 정보처리기사 시험 정보와 2026년 출제기준 PDF를 기준으로 과목명과 합격 기준을 정리했다.
- 후기 자료: 필기, 실기, 재도전/불합격 사례를 나누어 크롤했고, 각 배치는 별도 JSONL로 보존했다.
- 학습 반영 방식: URL과 요약 메타데이터는 남기되, 블로그 원문 문장은 그대로 복제하지 않았다.


## 9. 학습 전 검증 결과

학습 투입 전에는 최소 형식 검증과 내용 검증을 함께 본다. 여기서는 합본 JSONL이 파싱 가능한지, `messages` 구조가 있는지, assistant 응답이 JSON인지, 중복 ID가 없는지, 필기/실기 핵심 기준 문구가 포함되는지를 확인한다. 같은 기준은 `validate_exam_sft.py` CLI로도 재실행할 수 있게 코드화했다.


In [ ]:
validation_summary = {
    "rows": len(combined),
    "unique_ids": len({row.get("meta", {}).get("id") for row in combined}),
    "duplicate_ids": len(combined)
    - len({row.get("meta", {}).get("id") for row in combined}),
    "all_have_messages": all(
        isinstance(row.get("messages"), list) and len(row["messages"]) >= 3
        for row in combined
    ),
    "assistant_json_ok": True,
    "assistant_kind_values": Counter(),
    "missing_source_batch": 0,
    "missing_exam_type": 0,
}

for row in combined:
    meta = row.get("meta", {})
    validation_summary["missing_source_batch"] += int(not meta.get("source_batch"))
    validation_summary["missing_exam_type"] += int(not meta.get("exam_type"))
    assistant = row["messages"][-1]["content"]
    try:
        payload = json.loads(assistant)
    except json.JSONDecodeError:
        validation_summary["assistant_json_ok"] = False
        continue
    validation_summary["assistant_kind_values"][payload.get("kind")] += 1

validation_summary["assistant_kind_values"] = dict(
    validation_summary["assistant_kind_values"]
)
print(json.dumps(validation_summary, ensure_ascii=False, indent=2))

assert validation_summary["duplicate_ids"] == 0
assert validation_summary["all_have_messages"]
assert validation_summary["assistant_json_ok"]
assert validation_summary["missing_source_batch"] == 0
assert validation_summary["missing_exam_type"] == 0


In [ ]:
written_text = "\n".join(
    row["messages"][-1]["content"]
    for row in combined
    if row.get("meta", {}).get("exam_part") == "written"
)
practical_text = "\n".join(
    row["messages"][-1]["content"]
    for row in combined
    if row.get("meta", {}).get("exam_part") == "practical"
)

content_checks = {
    "written_mentions_subject_rule": "과목당 40점 이상" in written_text
    and "평균 60점 이상" in written_text,
    "written_mentions_exact_subject": "소프트웨어설계" in written_text,
    "practical_mentions_subject": "정보처리실무" in practical_text,
    "practical_mentions_pass_score": "60점 이상" in practical_text,
}
print(json.dumps(content_checks, ensure_ascii=False, indent=2))
assert all(content_checks.values())


In [ ]:
from sft_pipeline.build._archive_ipe.validate_exam_sft import (
    validate_samples as validate_exam_samples,
)

exam_validation_report = validate_exam_samples(combined_file)
print(json.dumps(exam_validation_report, ensure_ascii=False, indent=2))
assert exam_validation_report["ok"] == len(combined)
assert not exam_validation_report["errors"]


In [ ]:
from sft_pipeline.build.lib.validate_dataset import (
    validate_samples as validate_runtime_samples,
)

runtime_validation_report = validate_runtime_samples(runtime_file)
print(json.dumps(runtime_validation_report, ensure_ascii=False, indent=2))
assert runtime_validation_report["ok"] == len(runtime_samples)
assert not runtime_validation_report["errors"]


In [ ]:
training_validation_report = validate_runtime_samples(training_file)
print(json.dumps(training_validation_report, ensure_ascii=False, indent=2))
assert training_validation_report["ok"] == len(training_samples)
assert not training_validation_report["errors"]


### 현재 판정

`all_sft`는 정보처리기사 도메인 검수용 `phases` 스키마로 통과했고, `runtime_sft`는 기존 공통 `validate_dataset.py`까지 통과했다. 여기에 `follow_up` synthetic 100건, distractor 9건, runtime hardening 80건을 더해 최종 hardened split 234건도 공통 validator를 통과했다. 따라서 현재 정처기 데이터는 내부 학습 파이프라인에 섞을 수 있는 1차 시험 플랜 + 꼬리질문 + 경계 행동 + 구조화 출력 보강 데이터로 볼 수 있다.


### Phase 보존 방식

초기 구조화 데이터의 `phases`는 최종 런타임 출력 필드로 직접 남기지 않았다. 현재 서비스 런타임은 `summary_text/todos/calendar_events` 구조를 기대하므로, 각 phase의 task를 날짜 기준으로 펼쳐 기준일 당일 task는 `todos`, 미래 task는 `calendar_events`에 배치했다. phase명은 삭제하지 않고 각 task의 `tags`에 추가했으며, 변환 샘플에는 `meta.converted_from_schema = "exam-phases-v1"`를 기록했다.

즉 phase는 별도 필드가 아니라 일정 순서, C5 분기, task 태그로 투영된다. 대표 phase 태그는 `개념 1회독`, `기출·오답 누적`, `실전 마무리`, `기출 집중 회독`, `시험 직전 점검`, `D-1 최종 정리` 등이다.


## 10. Dry-Run 결과

정처기 내부 학습 후보 145건에 gold distractor 9건과 runtime hardening 80건을 섞어 실제 학습 직전 형태로 나누어 검증했다. 분할은 `provenance` 기준 stratified 방식으로 수행해 train/valid 양쪽에 plan, follow_up, distractor, hardening 샘플이 들어가도록 했다.


In [ ]:
dryrun_train_file = G / "ipe_hardened_dryrun_train.jsonl"
dryrun_valid_file = G / "ipe_hardened_dryrun_valid.jsonl"

dryrun_train = load_jsonl(dryrun_train_file)
dryrun_valid = load_jsonl(dryrun_valid_file)

dryrun_summary = {
    "train_rows": len(dryrun_train),
    "valid_rows": len(dryrun_valid),
    "train_provenance": dict(
        Counter(row.get("meta", {}).get("provenance", "?") for row in dryrun_train)
    ),
    "valid_provenance": dict(
        Counter(row.get("meta", {}).get("provenance", "?") for row in dryrun_valid)
    ),
    "train_assistant_kind": dict(Counter(assistant_kind(row) for row in dryrun_train)),
    "valid_assistant_kind": dict(Counter(assistant_kind(row) for row in dryrun_valid)),
}
print(json.dumps(dryrun_summary, ensure_ascii=False, indent=2))


In [ ]:
dryrun_train_report = validate_runtime_samples(dryrun_train_file)
dryrun_valid_report = validate_runtime_samples(dryrun_valid_file)
print(
    json.dumps(
        {"train": dryrun_train_report, "valid": dryrun_valid_report},
        ensure_ascii=False,
        indent=2,
    )
)
assert dryrun_train_report["ok"] == len(dryrun_train)
assert dryrun_valid_report["ok"] == len(dryrun_valid)
assert not dryrun_train_report["errors"]
assert not dryrun_valid_report["errors"]


### 로컬 학습 환경 점검

로컬 환경에서는 데이터 dry-run까지 통과했다. 다만 실제 Qwen LoRA 학습은 GPU와 학습 의존성이 필요하다. 현재 로컬 점검 결과 CUDA/MPS가 없고 `datasets`, `peft`, `unsloth`가 설치되어 있지 않아, 실제 모델 학습 dry-run은 RunPod 같은 GPU 환경에서 이어서 수행해야 한다.

RunPod에서는 최소 번들을 업로드한 뒤 아래 명령으로 짧은 학습 점검을 실행한다.

```bash
bash sft_pipeline/train/runpod_ipe_dryrun.sh
```

기본 설정은 0.10 epoch, `max_seq_len=2048`, batch 1, grad accumulation 4이다. dry-run의 성공 기준은 `SFTTrainer` 생성 통과, train loss 기록, adapter 저장, valid 평가 또는 후속 postcheck 실행 가능 여부다.

runpod gpu 환경: RTX 4090


## 11. 현재 한계와 다음 작업

현재 데이터셋은 정처기 전용 dry-run으로는 다음 실험을 진행할 수 있는 수준이다. `phases` 기반 검수본은 런타임 `summary_text/todos/calendar_events` 스키마로 변환했고, phase명은 task 태그로 보존했다. 또한 정보 부족 요청용 `follow_up` 100건, 경계 행동용 distractor 9건, 구조화 출력 오류를 겨냥한 hardening 80건을 추가했다. 이후 postcheck v2에서 plan/follow_up 라우팅 혼선, follow_up `question` 누락, C5/날짜/태그 문제가 확인되어 postcheck hardening 136건(plan 96, follow_up 40)을 추가했고, 최종 v2 split은 train 332건, valid 38건으로 구성했다.

권장 다음 작업:

1. `ipe_hardened_v2_dryrun_train/valid.jsonl`로 RunPod dry-run을 다시 실행한다.
2. `kind_eval.route_success_rate`, plan `parse_success_rate`, plan `consistency_success_rate`를 postcheck v2 이전 결과와 비교한다.
3. plan-only valid와 follow_up-only valid로 실패 축을 분리해 재평가한다.
4. v2에서 plan 정합성 성공률이 안정되면 epoch를 늘린 본 학습 후보를 만들고, validation 샘플 수를 늘려 과적합 여부를 확인한다.
5. 이후 SQLD, ADsP, 컴활 등 유사 자격증과 daily 계획 데이터를 섞어 시험 특화 능력과 범용 일정 생성 능력의 균형을 확인한다.


## 12. Hardening 반복 이력 (dry-run → v1 full)

postcheck 기반 반복 hardening과 epoch 증가를 통해 데이터셋·모델 품질을 개선했다.

### 12.1 이터레이션 요약표

| 이터레이션 | 학습 데이터 | epochs | route | follow_up route | plan route | plan consistency | 주요 발견 |
|---|---|---|---|---|---|---|---|
| distractor | 154건 | 0.10 | 0.60 | - | - | - | 구 postcheck, kind 미분류 |
| hardened v1 | 234건 | 0.10 | 0.40 | - | - | - | 날짜오타·영문태그 발생 |
| hardened v2 | 332건 | 0.10 | 0.65 | 0.75 | 0.58 | 0.25 | plan→follow_up 혼동, question 필드 누락 |
| hardened v3 | 494건 | 0.10 | 0.80 | 1.00 | 0.67 | 0.00 | follow_up 개선, JSON키 따옴표 누락, 날짜 환각 |
| **v1 full** | **494건** | **1.00** | **0.90** | **1.00** | **0.82** | **0.73** | **follow_up 완전 해결, C5 대폭 개선** |

### 12.2 v1 full 학습 결과 상세 (20샘플 평가)

- **EOS rate**: 95% — 거의 모든 출력이 정상 종료
- **전체 route 성공률**: 90% (18/20)
- **follow_up**: route 100%, parse 100% — 완전 해결
- **plan**: route 82%, parse 73%, consistency 73%
- **어댑터 크기**: 154MB (Qwen2.5-7B LoRA)

### 12.3 이터레이션별 데이터 전략

- **distractor**: `out_of_scope`, `chit_chat` 9건 추가
- **hardened v1**: runtime 45 + follow_up 100 + hardening 80 + distractor 9 = 234건
- **hardened v2**: v1 + postcheck_hardening 136건 = 370건
- **hardened v3**: v2 + v3_hardening 180건(C5 60, route 60, follow_up schema 60) = 550건
  - DATE_POOL 10개 날짜 순환으로 날짜 앵커링 방지
  - system prompt quality에 today 명시
  - repetition_penalty=1.05, max_new_tokens=512
- **v1 full**: v3 데이터 그대로, epoch 0.10 → 1.00으로 증가

### 12.4 남은 과제

1. **plan parse 27% 실패** — JSON 잘림(max_new_tokens 512 부족) + 날짜 연도 오타(`2-06-08`)
2. **plan route 18% 미스** — 충분한 정보에도 follow_up 응답하는 경우 잔존
3. **plan consistency 27% 실패** — C5 branch(todos = 오늘 날짜) 완전 습득 미완
4. **데이터 다양성** — 494건 중 실제 크롤 40건(8%), 정처기 단일 도메인

## 13. v2 학습 준비 및 결과 (2026-06-10)

> **요약**: v1 어댑터 HuggingFace 공개, v4 hardening 데이터 100건 추가, v2 학습 실행 및 평가, 연도 오타 문제 발견 및 수정, v2b 재학습 결과 확인.

---

### 13.1 v1 어댑터 HuggingFace 업로드

v1 full 학습으로 만들어진 LoRA 어댑터를 HuggingFace 모델 저장소에 업로드했다.

| 항목 | 내용 |
|---|---|
| 저장소 | `bigmooon/qwen2.5-7b-mongle-planner-ko-lora` (private) |
| 베이스 모델 | Qwen2.5-7B-Instruct |
| 어댑터 크기 | 162MB (`adapter_model.safetensors`) |
| 학습 데이터 | `ipe_hardened_v3_mix_sft.jsonl` 494건, 1.0 epoch |
| 커밋 메시지 | `feat: v1 full 1.0epoch LoRA adapter (route=90%, follow_up=100%, plan_consistency=73%)` |

> **HuggingFace란?**: AI 모델을 공유·저장하는 플랫폼. GitHub의 AI 모델 버전이라고 이해하면 된다. 학습된 어댑터를 여기에 올려두면 나중에 언제든 불러와 서비스에 연결할 수 있다.

---

### 13.2 v4 Hardening 데이터셋 구축

v1 postcheck 결과에서 발견된 3가지 실패 패턴을 겨냥해 신규 학습 데이터 100건을 추가 제작했다.

#### 실패 패턴별 추가 데이터

| 타겟 | 샘플 수 | 문제 | 해결 방법 |
|---|---|---|---|
| 연도 오타 수정 (`year_4digit`) | 30건 | `2-06-08` 같이 연도 앞 두 자리가 잘리는 현상 | 연도 4자리가 포함된 정상 예시 대량 추가 |
| C5 복합 기간 (`c5c_complex`) | 50건 | "오늘+이번 주", "오늘+내일" 요청 시 todos에 미래 날짜 혼입 | 복합 기간 표현에서도 todos=오늘만 임을 명시 |
| plan route 강화 (`route_plan_v2`) | 20건 | 충분한 정보가 있어도 follow_up으로 응답하는 잔존 오류 | 정보 충분 케이스 → plan 즉시 출력 예시 추가 |

#### 최종 v4 학습 데이터 구성

```
v3 mix (550건)
  + v4 hardening (100건)
  = ipe_hardened_v4_mix_sft.jsonl (650건)
    ├── train: 585건 (90%)
    └── valid: 65건 (10%)
```

| 출처 | 건수 | 설명 |
|---|---|---|
| distractor | 9 | 범위 밖·잡담 경계 케이스 |
| exam-crawl | 45 | 실제 정처기 후기 크롤 데이터 |
| exam-follow-up-synth | 200 | 정보 부족 시 되묻기(follow_up) 합성 |
| exam-synth | 396 | 시험 준비 플랜 합성 |

---

### 13.3 v2 학습 실행

**환경**: RunPod RTX 4090 / Qwen2.5-7B-Instruct 베이스 / 1.0 epoch

```bash
EPOCHS=1.0 bash sft_pipeline/train/runpod_ipe_dryrun.sh
```

---

### 13.4 v2 postcheck 결과 분석 (n=20)

> **postcheck란?**: 학습이 끝난 모델에게 평가용 문제 20개를 풀게 하고 정답률을 측정하는 자동 채점 시스템. EOS(정상 종료), 라우팅(어떤 응답 유형인지), 파싱(JSON 형식이 맞는지), 일관성(날짜 규칙 준수 여부)을 점검한다.

#### v1 → v2 비교표

| 평가 항목 | v1 (494건, 1.0ep) | v2 (650건, 1.0ep) | 변화 |
|---|---|---|---|
| **EOS 정상 종료율** | 95% | **100%** | ✅ 완전 해결 |
| **전체 라우팅 성공률** | 90% | **100%** | ✅ 완전 해결 |
| **plan JSON 파싱률** | 73% | **80%** | ↑ 개선 |
| **plan 일관성(C5 branch)** | 73% | ~53% | ↓ 저하 |

#### 결과 해석

**긍정적 변화:**
- `max_new_tokens` 512 → 768 수정으로 JSON 잘림 현상 해소 → 파싱 73%→80% 개선
- 라우팅 100% 달성: 충분한 정보가 있으면 follow_up 없이 plan 즉시 출력하는 동작 완전 학습
- EOS 100%: 모든 응답이 정상 종료

**문제 발견 — 연도 오타 악화:**

v4 hardening에서 사용자 메시지에 `"2026년"` 을 명시적으로 넣은 것이 역효과를 냈다.

```
기대: "due_date": "2026-06-20"
실제: "due_date": "2206-06-20"   ← 숫자 뒤바뀜
실제: "due_date": "22026-06-11"  ← 앞에 '2' 추가
```

모델이 "2026"을 강조하라는 신호를 system prompt와 user 메시지 두 곳에서 동시에 받자 오히려 연도를 과잉 생성하는 현상이 발생했다.

---

### 13.5 수정 조치 — v4b 데이터 재구성

**원인 분석:**

모델이 날짜를 학습하는 경로는 두 가지다:

```
[학습 경로 A]  system prompt: "기준일은 2026-06-10이다"
                                    ↓
               assistant 출력: {"due_date": "2026-06-10"}   ← 올바른 경로

[학습 경로 B]  user 메시지: "2026년 6월에 시험이야"
               + system prompt: "2026 연도 4자리 사용"
                                    ↓
               assistant 출력: {"due_date": "22026-06-10"}  ← 과잉 강조로 오염
```

**수정 방향:** user 메시지의 "2026년" 명시 제거. system prompt의 `기준일은 {오늘 날짜}` 하나만으로 연도를 학습하게 한다.

**변경 내용:**

| 변경 위치 | 변경 전 | 변경 후 |
|---|---|---|
| `YEAR_CASES` user 메시지 | `"2026년 6월에 정처기 필기..."` | `"정처기 필기 시험 7일 남았어..."` |
| `YEAR_SUFFIXES` | `" 연도를 2026으로 정확히 4자리로..."` | `" due_date는 YYYY-MM-DD 형식으로..."` |
| `_system()` extra_quality | `"due_date 연도는 반드시 4자리(2026)"` | `"due_date는 YYYY-MM-DD 형식"` |

> **핵심 원칙**: 연도 정보는 system prompt의 `기준일`에서 자연스럽게 습득하도록 유도. 프로덕션에서도 system prompt에 실제 오늘 날짜가 자동 주입되므로 이 방식이 더 자연스럽다.

**v4b 데이터셋 (재구성):**
- 동일 650건 / train 585건 / valid 65건
- user 메시지에서 연도 명시만 제거, 나머지 구조 동일

---

### 13.6 v2b postcheck 결과 (n=20, 2026-06-10)

> v4b 데이터(사용자 메시지에서 연도 명시 제거)로 재학습한 어댑터의 최종 자동 채점 결과 (n=20).

#### v2 → v2b 결과 비교

| 평가 항목 | v2 (n=20) | v2b (n=20) | 목표 | 달성 |
|---|---|---|---|---|
| **EOS 정상 종료율** | 100% | **100%** | 100% 유지 | ✅ |
| **전체 라우팅 성공률** | 100% | **100%** | 100% 유지 | ✅ |
| **plan JSON 파싱** | 80% | **80%** | 85%+ | ❌ |
| **plan 일관성(C5 branch)** | ~53% | **40%** | 75%+ | ❌ |
| **연도 오타** | 발생 (`22026-`) | **발생 (`2206-`, 3/20)** | 0건 | ❌ |

#### 실패 케이스 요약

- **parse_failures**: 9/20건 (45%)
- **연도 오염 확인 케이스**: index 3, 8, 15 (`2206-` 패턴 — 자릿수 뒤바뀜)
- 연도 오염이 없는 케이스도 C5 branch(todos=오늘 날짜) 미준수로 실패

> **패턴 관찰**: 연도 오염 방식이 v2의 `22026-`(앞에 '2' 추가)에서 `2206-`(자릿수 뒤바뀜)으로 변화했다. 사용자 메시지에서 연도를 제거하면 오염 방식이 달라지지만 근본 불안정성은 해소되지 않는다.

#### 결론

- EOS·라우팅은 완벽 유지 (100%)
- plan 일관성 53% → **40%로 오히려 저하** — v4b 수정이 C5 branch 학습에도 부정적 영향
- 연도 오타 패턴 변화에도 불구하고 15%(3/20) 잔존
- **v4b 접근(연도 제거)만으로는 불충분** — 더 강한 데이터 개입 필요

#### 다음 v3 어댑터 계획

| 과제 | 현황 | 대응 전략 |
|---|---|---|
| 연도 오타 (`2206-`) | 3/20 (15%) | 오류 형태 명시 교정 예시 50건+ 추가 |
| plan 일관성 (C5 branch) | 40% | todos=기준일 당일 규칙 케이스 대폭 추가 |
| plan parse | 80% | 현 수준 유지 목표 |

1. **v5 hardening 데이터 구축** — 연도 오타(50건) + C5 강화(80건) 추가
2. **v3 어댑터 학습** — RunPod RTX 4090 / 1.0 epoch
3. **n=20 full postcheck 목표**: plan 일관성 75%+, 연도 오타 0건

## 14. v3 어댑터 결과 및 데이터셋 정리 (2026-06-10)

### 14.1 v3 어댑터 결과 (n=20) — 폐기

v5 hardening 데이터(연도 오타 50건 + C5 강화 80건 = 130건)를 추가한 v5 mix(780건)로 학습한 결과.

| 평가 항목 | v2b (기준) | v3 | 변화 |
|---|---|---|---|
| **EOS 정상 종료율** | 100% | 100% | — |
| **전체 라우팅 성공률** | 100% | 100% | — |
| **plan JSON 파싱률** | 80% | **58.8%** | ↓ 악화 |
| **plan 일관성(C5 branch)** | 40% | 악화 | ↓ |

**원인**: v5 hardening에서 `summary_text`에 오늘 날짜를 명시(`"오늘은 2026-06-10이므로..."`)한 것이 역효과. 모델이 날짜 문자열 복사 과정에서 파싱 실패율 증가.

> **결론**: 연도 오타는 tokenizer 불안정성에서 기인하며 SFT 데이터 조작만으로 근본 해결 불가.
> **v2b 어댑터를 현재 최선 모델로 확정**. API 레이어 날짜 후처리로 대응.

---

### 14.2 데이터셋 정리

v3 결과 악화로 관련 중간 산출물 삭제 (39MB → 5.2MB).

| 삭제 파일 | 이유 |
|---|---|
| `ipe_hardened_v5_*` (3건) | v3 어댑터 학습용 — 결과 악화로 폐기 |
| `exam_ipe_v5_hardening_sft.jsonl` | v5 hardening 원본 — v3 폐기에 따라 삭제 |
| `ipe_hardened_v3_*`, `exam_ipe_v3_hardening_sft.jsonl` | 구버전 — v4로 대체됨 |
| `ipe_hardened_v2_*` | 구버전 — v4로 대체됨 |
| `ipe_hardened_*` (버전 없음) | 초기 실험 — 구버전 |
| `ipe_distractor_*` | 디스트랙터 실험 — v4 mix에 통합됨 |
| `exam_information_processing_engineer_*` (구형 빌드 산출물) | 구버전 |
| `distractor_gold_sft.jsonl` | 초기 실험 |

**보존:**
- `ipe_hardened_v4_*` — v2b 어댑터 학습에 사용된 최종 데이터
- `crawl_results_*`, `urls_*`, `raw_cases_*` — 원본 크롤 데이터 (재생성 불가)